# Reading input data {#sec-input-memb}

The objective of this section is to read input data in different formats and create a summary of the relationships in a consistent output format.

Input formats are convenient for data entry, but not optimal for data analysis.

Different experts will work with different types of input formats.

A consistent output format is useful for designing streamlined operations and functions. This format allows better interoperability in data operations. 

The output format can be back-transformed to the input formats if required.


## Set up

We start a session in _R_ and load the libraries we need for this example

In [1]:
library(readr)
library(readxl)
library(dplyr)
library(tidyr)
library(stringr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




We also want to declare the relative path, for example using the package `here`:

In [2]:
here::i_am("read-input-xref-data.ipynb")

here() starts at /Users/z3529065/proyectos/EIW/TASVEG-XREF



### Functions

We declare two functions that we will use to reshape input data to long format membership tables.

For example, we want a function that reshapes the data from a wide format to a long format membership table.

In [3]:
pivot_wide_membership_table <- function(xref,
                            original_names,
                            target_names,
                            membership_values,
                            expert_name = NULL,
                            target = NULL) {
    if (!is.null(target)) {
       xref <-  dplyr::filter(xref, 
        dplyr::if_any(dplyr::any_of(target_names), ~ .x %in% target))
    }
    
    memb_table <- xref |>
        dplyr::select(
            dplyr::any_of(original_names),
            dplyr::any_of(target_names),
            dplyr::any_of(membership_values)
        ) |> 
    tidyr::pivot_longer(
        dplyr::any_of(c(target_names,membership_values)),
        cols_vary = "slowest",
        names_to = c(".value"),
        names_pattern = "([[:alpha:]]+)") 

    if (!is.null(expert_name)) {
       memb_table <-  mutate(memb_table, expert = expert_name) 
    }
    return(memb_table)
}


::: {.aside}
This will be replaced by functions from a package under development.
:::

Or a function that reshapes de data from a matrix to a long format membership table.

In [4]:
pivot_membership_matrix <- function(xref,
                            original_names,
                            target_names,
                            expert_name = NULL,
                            target = NULL) {
    if (!is.null(target)) {
       xref <-  dplyr::filter(xref, 
        dplyr::if_any(dplyr::any_of(target_names), ~ .x %in% target))
    }
    memb_table <- xref  |> 
    tidyr::pivot_longer(
        dplyr::any_of(target_names),
        names_to="efg",
        values_to="memb")
    if (!is.null(expert_name)) {
        memb_table <-  mutate(memb_table, expert = expert_name) 
        }
        return(memb_table)
}


::: {.aside}
This will be replaced by functions from a package under development.
:::

## Read input tables prepared by experts

We have cross-referencing tables from four experts, we identify the expert by their initials: JB, AL, AK, and MGT.

Experts use different formats to represent the the cross-referencing information.

### Expert JB

Input data is in excel format, we will use function `read_excel` from package `readxl`:

In [5]:
xref_expert_JB <- read_excel(here::here("input-XREFs/Alignments-TASVEGtoGET-Revised-by-Jayne-2026-06-26.xlsx"), "tas_cr_100626_REV-JB")


New names:
• `` -> `...1`


Let's have a glimpse at the data:

In [6]:
glimpse(xref_expert_JB)

Rows: 156
Columns: 14
$ ...1                                                <dbl> 52, 98, 49, 114, 1…
$ VEG_GROUP                                           <chr> "Saltmarsh and wet…
$ VEGCODE                                             <chr> "AAP", "AHF", "AHL…
$ VEGCODE_D                                           <chr> "(AAP) Alkaline pa…
$ Agroup                                              <chr> "TF1.7", "F2.3", "…
$ `AMembership-estimate`                              <dbl> 1.00, 0.60, 0.55, …
$ Bgroup                                              <chr> NA, "F2.2", "TF 1.…
$ `B-estimate`                                        <dbl> NA, 0.25, 0.30, NA…
$ Cgroup                                              <chr> NA, "F3.1", "TF 1.…
$ `C-estimate`                                        <dbl> NA, 0.10, 0.10, NA…
$ Dgroup                                              <chr> NA, "TF1.3", "MFT1…
$ `D-estimate`                                        <dbl> NA, 0.05, 0.05, NA…
$ `Notes on ecolog

We use the function `rename` to change the names of the columns, the function will work best if we follow this naming convention.

:::{.aside}
This renaming step could be integrated into the function.
:::

In [7]:
xref_expert_JB_renamed <- 
    xref_expert_JB |>
    rename(
        efg_1 = "Agroup", efg_2 = "Bgroup", efg_3 = "Cgroup", efg_4 = "Dgroup",
        memb_1 = `AMembership-estimate`, memb_2 = `B-estimate`, memb_3 = `C-estimate`, memb_4 = `D-estimate`,
    ) 

and then use our custom function `pivot_wide_membership_table` to create an output in long format.

In [8]:
xtab1 <- xref_expert_JB_renamed |>
  pivot_wide_membership_table(
    original_names=c("VEGCODE"),
    target_names=sprintf("efg_%s",1:4),
    membership_values=sprintf("memb_%s",1:4),
    expert_name = "JB"
  ) |>
  filter(!is.na(efg)) |>
  arrange(VEGCODE)

Let's have a glimpse at the output:

In [9]:
glimpse(xtab1)

Rows: 274
Columns: 4
$ VEGCODE <chr> "AAP", "AHF", "AHF", "AHF", "AHF", "AHL", "AHL", "AHL", "AHL",…
$ efg     <chr> "TF1.7", "F2.3", "F2.2", "F3.1", "TF1.3", "TF1.4", "TF 1.7", "…
$ memb    <dbl> 1.00, 0.60, 0.25, 0.10, 0.05, 0.55, 0.30, 0.10, 0.05, 1.00, 0.…
$ expert  <chr> "JB", "JB", "JB", "JB", "JB", "JB", "JB", "JB", "JB", "JB", "J…


The output has four columns:

- `VEGCODE` is the original code of the units used in the TASVEG 5.0 dataset
- `efg` is the code/name for the target Ecosystem Functional Group
- `memb` is the membership value between 0 and 1
- `expert` is the name (initials) of the expert

We will bring all data to the same format of four columns

### Expert MGT

Input data is in `CSV` format, we will use funciton `read_csv` from the `readr` package. We get some warnings about names for the columns, then the column specification information.

In [10]:
xref_expert_MGT <- read_csv(here::here("input-XREFs/tasveg_lut_GET3_MGT230426.csv"))

New names:
• `` -> `...1`
• `` -> `...13`
• `` -> `...14`
Rows: 156 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (8): VEG_GROUP, VEGCODE, VEGCODE_D, Notes, GET1, GET2, GET3, ...13
dbl (5): ...1, GET1_p, GET2_p, GET3_p, ...14
lgl (1): workflow

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


We will use the same combination of `rename` and `pivot_wide_membership_table` as before, but adjust the column names based on the message produced by `read_csv`. (We could also use `glimpse` as before).

In [11]:
xtab2 <- xref_expert_MGT |>
  rename(
    efg_1="GET1",efg_2="GET2",efg_3="GET3",
    memb_1=`GET1_p`,memb_2=`GET2_p`,memb_3=`GET3_p`
) |>
pivot_wide_membership_table(
  original_names=c("VEGCODE"),
  target_names=sprintf("efg_%s",1:4),
  membership_values=sprintf("memb_%s",1:4)
) |>
  filter(!is.na(efg)) |>
  mutate(expert="MGT") |>
  arrange(VEGCODE)

We can confirm the output is in the same format:

In [12]:
glimpse(xtab2)

Rows: 261
Columns: 4
$ VEGCODE <chr> "AAP", "AAP", "AHF", "AHF", "AHL", "AHS", "ARS", "ASF", "ASF",…
$ efg     <chr> "TF1.7", "TF1.6", "F2.3", "TF1.3", "F2.3", "MFT1.3", "TF1.3", …
$ memb    <dbl> 0.7, 0.3, 0.7, 0.3, 1.0, 1.0, 1.0, 0.6, 0.4, 1.0, 1.0, 0.5, 0.…
$ expert  <chr> "MGT", "MGT", "MGT", "MGT", "MGT", "MGT", "MGT", "MGT", "MGT",…


### Expert AK

Expert AK uses again the `excel` format and we see a message about aparently duplicated column names. Also it appears the column names are very long.

In [13]:
xref_expert_AK <- read_excel(here::here("input-XREFs/GET_TASVEG_NVIS_Crosswalk_AK-final_050726.xlsx"))

New names:
• `T2.3 Oceanic cool temperate rainforests` -> `T2.3 Oceanic cool temperate
  rainforests...21`
• `T2.3 Oceanic cool temperate rainforests` -> `T2.3 Oceanic cool temperate
  rainforests...22`


We need a `glimpse` again:

In [14]:
glimpse(xref_expert_AK)

Rows: 156
Columns: 39
$ Location                                                 <chr> "Tasmania", "…
$ `TASVEG VEGCODE`                                         <chr> "AAP", "AHF",…
$ `GET L3 EFG`                                             <lgl> NA, NA, NA, N…
$ `F1.1 Permanent upland streams`                          <dbl> 0, 0, 0, 0, 0…
$ `F1.2 Permanent lowland rivers`                          <dbl> 0, 0, 0, 0, 0…
$ `F1.4 Seasonal upland streams`                           <dbl> 0, 0, 0, 0, 0…
$ `F1.5 Seasonal lowland rivers`                           <dbl> 0, 0, 0, 0, 0…
$ `F2.1 Large permanent freshwater lakes`                  <dbl> 0, 0, 0, 0, 0…
$ `F2.2 Small permanent freshwater lakes`                  <dbl> 0.0, 0.4, 0.0…
$ `F2.3 Seasonal freshwater lakes`                         <dbl> 0.0, 0.4, 0.0…
$ `F3.1 Large reservoirs`                                  <dbl> 0, 0, 0, 0, 0…
$ `F3.2 Constructed lacustrine wetlands`                   <dbl> 0, 0, 0, 0, 0…
$ `F3.3 Rice paddi

The format is different, here the membership values are stored in a matrix format where the original unit is represented in rows, and the target units in columns.

We will need a different approach here, first let's get a vector of the target units:

In [15]:
efg_cols <- colnames(xref_expert_AK)[grepl("[MFT0-9]+.[0-9]+", colnames(xref_expert_AK))]

Now we can use `pivot_longer` in this way.

In [16]:
xtab3 <- pivot_membership_matrix(
    xref_expert_AK,
    target_names =efg_cols,
    expert_name = "AK") |>
  select(
    VEGCODE=`TASVEG VEGCODE`, 
    efg, 
    memb,
    expert) |>
  filter(memb>0)

Let's check we got the output right:

In [17]:
glimpse(xtab3)

Rows: 216
Columns: 4
$ VEGCODE <chr> "AAP", "AHF", "AHF", "AHF", "AHL", "AHS", "ARS", "ASF", "ASP",…
$ efg     <chr> "TF1.7 Boreal and temperate fens", "F2.2 Small permanent fresh…
$ memb    <dbl> 1.00, 0.40, 0.40, 0.20, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.…
$ expert  <chr> "AK", "AK", "AK", "AK", "AK", "AK", "AK", "AK", "AK", "AK", "A…


It looks good, only difference is that instead of EFG alphanumeric codes, we actually have full EFG names. We will deal with that later.

### Expert AL

Uses the same format as the previous expert, so we basically repeat the same steps:

In [18]:

xref_expert_AL <- read_excel(here::here("input-XREFs/GET_TASVEG_NVIS_Crosswalk_AL-final_050726.xlsx"))


In [19]:
efg_cols <- colnames(xref_expert_AL)[grepl("[MFT0-9]+.[0-9]+",colnames(xref_expert_AL))]

xtab4 <- pivot_membership_matrix(
    xref_expert_AL,
    target_names =efg_cols,
    expert_name = "AL") |>
  select(
    VEGCODE=`TASVEG VEGCODE`, 
    efg, 
    memb,
    expert) |>
  filter(memb>0)

## Combine all data

We now have four tables (`xtab1`, `xtab2`, `xtab3`, and `xtab4`).

We use `bind_rows` to combine them, we `arrange` rows by `VEGCODE` and `expert`, and then we apply some string opperations from package `stringr` to replace the long EFG names with the shorter EFG codes. 

In [20]:
xref_table_experts <- 
  bind_rows(xtab1, xtab2, xtab3, xtab4) |>
  arrange(VEGCODE,expert) |>
  mutate(efg=str_extract(efg,"[MFT\ 0-9]+.[0-9]+") |>
    str_remove_all(" "))

Let's see this full table:

In [ ]:
xref_table_experts

This is a selection of vegetation communities dominated by Eucalyptus amygdalina

In [38]:

xref_table_experts |>
    filter(grepl("^DA",VEGCODE)) 


VEGCODE,efg,memb,expert
<chr>,<chr>,<dbl>,<chr>
DAC,T2.6,1.0,AK
DAC,T2.6,1.0,AL
DAC,T2.6,1.0,JB
DAC,T2.6,0.9,MGT
DAC,T3.2,0.1,MGT
DAD,T2.6,1.0,AK
DAD,T2.6,1.0,AL
DAD,T2.6,0.9,JB
DAD,T4.4,0.1,JB


This is a selection of vegetation communities belonging to Native grasslands vegetation group

In [28]:
xref_table_experts |>
filter(grepl("^G",VEGCODE))

VEGCODE,efg,memb,expert
<chr>,<chr>,<dbl>,<chr>
GCL,T4.5,0.8,AK
GCL,T7.5,0.2,AK
GCL,T4.5,0.8,AL
GCL,T7.5,0.2,AL
GCL,T4.5,0.6,JB
GCL,T7.5,0.4,JB
GCL,T4.4,0.5,MGT
GCL,T4.5,0.3,MGT
GCL,T7.5,0.2,MGT


In [ ]:
We save this output for later:

In [37]:
output_dir <- here::here("output-XREFs")
if (!dir.exists(output_dir))
  dir.create(output_dir)

write_csv(
  xref_table_experts,
  file = here::here(output_dir,"First-round-output.csv")
)

interpreters <- c("Worrysome Wren","Frightened Flamingo","Pointless Panda","Tantalising Tarantula")
names(interpreters) <- c("AL","AK","JB","MGT")

xref_table_experts |>
transmute(
    VEGCODE,
    efg,
    memb,
    interp=interpreters[expert],
    round=1
) |>
write.csv(here::here(output_dir,"TASVEG-xref-first-round-anon.csv"))

## Summarise the data

Now that we have the input from four experts in the same format, we can produce some informative summaries:

In [ ]:
(candidates_per_vegcode <- xref_table_experts |>
  group_by(VEGCODE) |>
  summarise(`Nr. of candidates`=n_distinct(efg), `Highest membership`=max(memb)))

# A tibble: 156 × 3
   VEGCODE `Nr. of candidates` `Highest membership`
   <chr>                 <int>                <dbl>
 1 AAP                       2                  1  
 2 AHF                       4                  0.7
 3 AHL                       5                  1  
 4 AHS                       1                  1  
 5 ARS                       3                  1  
 6 ASF                       2                  1  
 7 ASP                       1                  1  
 8 ASS                       1                  1  
 9 AUS                       2                  1  
10 AWU                       5                  1  
# ℹ 146 more rows
# ℹ Use `print(n = ...)` to see more rows

In [ ]:
candidates_per_vegcode |>
  pull(`Nr. of candidates`) |>
  table()


 1  2  3  4  5  9 
35 68 31 13  8  1 

In [ ]:
xref_table_experts |>
  group_by(VEGCODE, efg, memb) |>
  summarise(`Nr. of experts`=n_distinct(expert), 
    experts=paste(expert,collapse=";"),
    .groups="drop") |>
  arrange(VEGCODE, desc(`Nr. of experts`), desc(memb))

# A tibble: 631 × 5
   VEGCODE efg    memb `Nr. of experts` experts 
   <chr>   <chr> <dbl>            <int> <chr>   
 1 AAP     TF1.7 1                    3 AK;AL;JB
 2 AAP     TF1.7 0.7                  1 MGT     
 3 AAP     TF1.6 0.3                  1 MGT     
 4 AHF     F2.3  0.4                  2 AK;AL   
 5 AHF     F2.3  0.7                  1 MGT     
 6 AHF     F2.3  0.6                  1 JB      
 7 AHF     F2.2  0.4                  1 AK      
 8 AHF     F2.2  0.333                1 AL      
 9 AHF     TF1.3 0.333                1 AL      
10 AHF     TF1.3 0.3                  1 MGT     
# ℹ 621 more rows
# ℹ Use `print(n = ...)` to see more rows

In [ ]:

xref_expert_JB |>
  filter(VEGCODE %in% "AAP")

# A tibble: 1 × 14
   ...1 VEG_GROUP     VEGCODE VEGCODE_D Agroup `AMembership-estimate` Bgroup `B-estimate` Cgroup `C-estimate`
  <dbl> <chr>         <chr>   <chr>     <chr>                   <dbl> <chr>         <dbl> <chr>         <dbl>
1    52 Saltmarsh an… AAP     (AAP) Al… TF1.7                       1 NA               NA NA               NA
# ℹ 4 more variables: Dgroup <chr>, `D-estimate` <dbl>, `Notes on ecological types in mapping unit` <chr>,
#   `Note on separating mapping units into GET classes` <chr>

In [ ]:
xref_expert_ALL |>
  filter(TASVEG %in% "AAP")

# A tibble: 1 × 56
  TASVEG GET_Level3 GET3_w  F2.8 TF1.6 TF1.7  F2.2  F2.3  F2.4  F3.1 TF1.3 MFT1.3
  <chr>  <chr>       <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl> <dbl>  <dbl>
1 AAP    TF1.7        0.94  0.04  0.02  0.94    NA    NA    NA    NA    NA     NA
# ℹ 44 more variables: `TF 1.3` <dbl>, `TF 1.7` <dbl>, TF1.4 <dbl>, TF1.5 <dbl>,
#   T2.6 <dbl>, T4.4 <dbl>, T3.2 <dbl>, T2.5 <dbl>, TF1.2 <dbl>, TF16 <dbl>,
#   T7.2 <dbl>, T7.5 <dbl>, T7.1 <dbl>, T7.3 <dbl>, MT2.1 <dbl>, MT3.1 <dbl>,
#   T3.1 <dbl>, T7.4 <dbl>, F3.3 <dbl>, T4.5 <dbl>, T6.4...33 <dbl>,
#   T6.4...34 <dbl>, TF7.2 <dbl>, `TF 1.6` <dbl>, `TF 4.5` <dbl>, T2.3 <dbl>,
#   TF2.3 <dbl>, `TF 1.2` <dbl>, F1.1 <dbl>, F1.2 <dbl>, F1.4 <dbl>, F1.5 <dbl>,
#   F2.1 <dbl>, F3.2 <dbl>, FM1.2 <dbl>, MT1.1 <dbl>, MT1.4 <dbl>, T3.4 <dbl>, …

In [ ]:
xref_expert_AL |>
  filter(`TASVEG VEGCODE` %in% "AAP")
xref_expert_AK |>
  filter(`TASVEG VEGCODE` %in% "AAP")

# A tibble: 1 × 39
  Location `TASVEG VEGCODE` `GET L3 EFG`
  <chr>    <chr>            <lgl>       
1 Tasmania AAP              NA          
# ℹ 36 more variables:
#   `F1.1 Permanent upland streams` <dbl>,
#   `F1.2 Permanent lowland rivers` <dbl>,
#   `F1.4 Seasonal upland streams` <dbl>,
#   `F1.5 Seasonal lowland rivers` <dbl>,
#   `F2.1 Large permanent freshwater lakes` <dbl>,
#   `F2.2 Small permanent freshwater lakes` <dbl>, …

In [ ]:
xref_expert_ALL <- read_csv(here::here("input-XREFs/GET_Level3_LUT_120826.csv"))

New names:
• `T6.4` -> `T6.4...33`
• `T6.4` -> `T6.4...34`
Rows: 157 Columns: 56
── Column specification ─────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): TASVEG, GET_Level3
dbl (53): GET3_w, F2.8, TF1.6, TF1.7, F2.2, F2.3, F2.4, F3.1, TF1.3, MFT1.3, ...
lgl  (1): NA

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
efg_cols <- colnames(xref_expert_ALL)[grepl("[MFT0-9]+.[0-9]+",colnames(xref_expert_ALL))]
pivot_longer(xref_expert_ALL,cols=any_of(efg_cols),names_to="efg",values_to="memb") |>
  transmute(VEGCODE=TASVEG, efg, memb, expert="aggregated") |>
  filter(!is.na(memb))

# A tibble: 403 × 4
   TASVEG efg    memb expert    
   <chr>  <chr> <dbl> <chr>     
 1 AAP    F2.8  0.04  aggregated
 2 AAP    TF1.6 0.02  aggregated
 3 AAP    TF1.7 0.94  aggregated
 4 AHF    F2.2  0.35  aggregated
 5 AHF    F2.3  0.4   aggregated
 6 AHF    F2.4  0.06  aggregated
 7 AHF    F3.1  0.02  aggregated
 8 AHF    TF1.3 0.17  aggregated
 9 AHL    F2.3  0.06  aggregated
10 AHL    TF1.3 0.642 aggregated
# ℹ 393 more rows
# ℹ Use `print(n = ...)` to see more rows